<a href="https://colab.research.google.com/github/d-noe/NLP_DH_F26/blob/main/code/2_word_representation/Hands_on_2_word2vec_semantic_shift.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hands-on Session 2: Word2Vec & Semantic Shift

![](https://thumb.wikimedia.org/wikipedia/commons/thumb/9/96/Turner_-_Rain%2C_Steam_and_Speed_-_National_Gallery_file.jpg/960px-Turner_-_Rain%2C_Steam_and_Speed_-_National_Gallery_file.jpg?utm_source=commons.wikimedia.org&utm_campaign=index&utm_content=thumbnail)
<p align="right">
  <i>Rain, Steam and Speed – The Great Western Railway</i>. J. M. W. Turner. (1844). Oil on canvas.
</p>


> 💡 **Reference & Inspiration:** This notebook is directly based on the paper *"Machines in the media: semantic change in the lexicon of mechanization in 19th-century British newspapers"* [(Pedrazzini & McGillivray, 2022)](https://aclanthology.org/2022.nlp4dh-1.12.pdf), its companion [code base](https://github.com/Living-with-machines/DiachronicEmb-BigHistData), as well as [released artefacts](https://zenodo.org/records/7887305).
> This article was written in the context of the broader [Living with Machines](https://livingwithmachines.ac.uk/) project.

Words are not set in stone. As time goes by, society changes, the meanings, associations, and connotations of words evolve along side it. In linguistics, this dynamical process is usually refered to as [***semantic shift***](https://en.wikipedia.org/wiki/Semantic_change) (or semantic change).

Today, we are going to see how NLP methods, and in particular `word2vec` embeddings, can help us study the evolution of language, through *semantic shifts* within a corpus of British newspapers from the XIXth century.

Starting from a set of raw (pre-trained) word2vec vector representations (released by [(Pedrazzini & McGillivray, 2022)](https://aclanthology.org/2022.nlp4dh-1.12.pdf)), we will explore semantic spaces, gain hands-on experience in handling such objects, and finally, you will be able to compute and observe *semantic shifts* from the data.

Talking about semantic shift throughout the industrial revolution, how did the distributional representation of the word `train` evolve?




<a name="train_plot"></a>

In [ ]:
#@title Here is an example of what you can expect by the end of the notebook! 🚂


from IPython.core.display import display, HTML
import urllib.request

fp = urllib.request.urlopen("https://github.com/d-noe/NLP_DH_F26/raw/refs/heads/main/props/w2v_train_viz.html")
mybytes = fp.read()

mystr = mybytes.decode("utf8")
fp.close()

display(HTML(mystr))

**Table of content**:

1. [Preliminaries](#preliminaries)
2. [Loading Representations](#load)
3. [Aligning Representations](#align)
4. [Analysis](#analysis)

<a name="preliminaries"></a>

# Preliminaries

In this notebook, we will start from **pre-trained diachronic word-vector representations** from [Pedrazzini & McGillivray (2022)](https://aclanthology.org/2022.nlp4dh-1.12.pdf) (you won't need to train the word2vec models yourself), but before we dive into analyzing vectors, let's look under the hood at how these representations were created.
Indeed, understanding the data lineage is key to interpreting your results critically.

The complete processing pipleine behind our vectors can be decomposed as follows:

`Primary Source (Newspapers)` $\to$ `Press Archival` $\to$ `Digitization` $\to$ `OCR` $\to$ `Corpus Preprocessing` $\to$ `Corpus splits` $\to$ `Train word2vec`

A coarse description of the main steps is provided below.

### 1. The Raw Material (Archival Press)
The source material consists of two large historical newspaper collections spanning the 19th century (1801–1920):
* **Heritage Made Digital** (British Library): ~2.3 billion tokens across 12 title series.
* **Living with Machines**: ~2.3 billion tokens across 107 title series.

Together, this gives us a massive corpus of **~4.6 billion tokens** documenting daily life, industrial change, and evolving language in Britain.

### 2. Digitization & OCR Processing
Historical newspapers were photographed and processed via Optical Character Recognition (OCR) to convert physical print into machine-readable text.

> 🔍 **Reality Check:** 19th-century OCR is inherently noisy! Damaged paper, smudged ink, non-standard typography, and archaic spellings mean that OCR errors inevitably slip into the raw text. Also, think about the conservation/degradation of the base material, **would you expect the OCR quality to be consistent across the whole corpus?**

### 3. Post-OCR & Preprocessing

To make this noisy text suitable for training vector space models, the corpus went through the following text-cleaning steps:
* **OCR repair & formatting:** Removing newline characters, line-break hyphenations, word breaks caused by OCR, and punctuation.
* **Standard NLP normalization:** Lowercasing all tokens and filtering out English stopwords (via NLTK).

### 4. Temporal Slicing

To track semantic change diachronically, the cleaned corpus was segmented into **10-year time slices** (1800s, 1810s, ..., 1910s). This allows us to observe how words moved through semantic space decade by decade.

Note that in the context of this notebook, we selected only every other decade (1800s, 1820s, 1840s, 1860s, 1880s, and 1900s).

### 5. Training Diachronic Word2Vec Models

An independent **Word2Vec** model was trained for each decade using [`gensim`](https://radimrehurek.com/gensim/models/word2vec.html).
Based on hyperparameter optimization via grid search, the original authors selected:
* **Architecture:** Skip-gram with Negative Sampling (SGNS).
* **Vector Dimension:** $D = 200$.
* **Context Window:** $3$.
* **Minimum Word Frequency:** $1$ (retaining low-frequency terms before pruning).
* **Epochs:** $5$.

**Curious about the training pipeline?**

You can check the original code used to train these `word2vec` modesl in the authors' repository: [`Living-with-machines/DiachronicEmb-BigHistData/blob/main/scripts/training/train_diach_emb.py`](https://github.com/Living-with-machines/DiachronicEmb-BigHistData/blob/main/scripts/training/train_diach_emb.py).

If you want to train you own model, remember that `gensim` makes it pretty simple once you have the corpus. A `word2vec` model can be trained using only the following code:

```python
from gensim.models import Word2Vec

# Load corpus, split sentences, tokenize,
# Define parameters, ...
# [ ... ]

model = Word2Vec( # instantiate & trains model
  sentences=sentences, # list[list] of tokenized sentences
  size=embedding_dimension, # size of the embeddings
  sg=1, # skipgram rather than cbow
  window=window_size, # size of the window
  epochs=n_epochs, # number of training passes
)
```

No further ado, let's jump into code!

<a name="load"></a>

# Loading raw pre-trained vector representations

Now that we know where our data comes from, let's prepare our coding environment:

1. installing and loading the libraries that we will need throughout this notebook
2. loading the word-vectors representation files ([stored on the GitHub repository](https://github.com/d-noe/NLP_DH_F26/tree/main/data/lwm_wordvecs)).

In [ ]:
!pip install gensim
!pip install adjustText
!pip install altair

In [ ]:
# To load the pretrained vectors from GitHub
import os
import requests
import tempfile

# Manipulate gensim's word2vec vectors
from gensim.models import KeyedVectors

# General data manipulation & computation
import numpy as np
import pandas as pd
from scipy import spatial # for cosine distance

# Visualisation
import matplotlib.pyplot as plt
import altair as alt # interactive chart
# dimensionality reduction
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

In [ ]:
# Helper function to load gensim KeyedVector stored on GitHub
def load_kv_from_github(
    file_name:str,
    vectors_folder:str="data/lwm_wordvecs/",
    base_url:str="https://raw.githubusercontent.com/d-noe/NLP_DH_F26/main/",
):
    # Get the file from the url
    url = os.path.join(base_url, vectors_folder, file_name)
    response = requests.get(url)
    response.raise_for_status()

    # Create a temporary file
    fd, path = tempfile.mkstemp(suffix=".bin")

    try:
        # Close the file descriptor so Gensim can open it normally
        os.close(fd)

        # Write the downloaded data
        with open(path, "wb") as f:
            f.write(response.content)

        # Now load it
        kv = KeyedVectors.load_word2vec_format(path,binary=True)

    finally:
        # Clean up the temporary file
        if os.path.exists(path):
            os.remove(path)

    # Return the vectors
    return kv

In [ ]:
# The decades we have data for (also the names of the associated files)
decades = ['1800s', '1820s', '1840s', '1860s', '1880s', '1900s']

Now that everything is ready, we can load the data. We will the vector representations associated with each decade and store it in a dictionary with the format:

```python
loaded_kvs = {
  "<DECADE_IDENTIFIER>": gensim.KeyedVectors
}
```

In [ ]:
loaded_kvs = {
    y: load_kv_from_github(file_name=f"pruned_{y}.bin")
    for y in decades
}

We can now check that we loaded the data successfully. We will print the number of word-vectors for each decade, and the dimension ($d$) of the vectors:

In [ ]:
for k, v in loaded_kvs.items():
  print(f"Vectors for decade {k} contain {len(v):,} word-vectors of dimensionality d = {v.vector_size}.")

## General exploration

Everything is ready, we can now familiarize with the data.

The following subsection walks you through the **visualisation** of the vectors stored for **one specific model** (decade). You can also use this space to explore further the data: look at the vocabulary (`KeyedVectors.key_to_index`), find most similar word (`KeyedVectors.most_similar`), etc.

We will then focus on computing [**diachronic cosine similarity** scores](#cos_sim_1).

### Plot the word-vectors for one specific time slice

The following cells show how to visualise the semantic space of one specific model. We will first filter the words to include in the visualisation (here based on most common English words based on Wikipedia(!)), and reduce the dimensionality of the selected vectors from $d$ to $2$ dimensions!

In [ ]:
# select the model you like!

timeslice_to_plot = "1900s"
most_common_english = 2000

In [ ]:
# Load most common English word to represent
url = "https://gist.githubusercontent.com/h3xx/1976236/raw/bbabb412261386673eff521dddbe1dc815373b1d/wiki-100k.txt"
response = requests.get(url)
response.raise_for_status()
wiki_100k_text = response.text
wiki_100k = [w.lower() for w in wiki_100k_text.split('\n') if (not w.startswith('#'))]


In [ ]:
# find the words in the vocabulary AND in the most common English vocabulary
words_to_plot = list(set(wiki_100k[:most_common_english]).intersection(set(loaded_kvs[timeslice_to_plot].key_to_index.keys())))

print(f"Original number of words in the vocabulary: {len(loaded_kvs[timeslice_to_plot]):,}.")
print(f"Number of word-vectors to be included in the plot: {len(words_to_plot):,}.")

In [ ]:
# retrive the vectors associated with the selected words

vectors_to_plot = [
    loaded_kvs[timeslice_to_plot][w] # gets the vector representation from the model stored in the dict
    for w in words_to_plot
]

In [ ]:
# Instantiate dimensionality reducer
reducer = TSNE(
    n_components = 2, # 2D
    perplexity = 5, # check documentation
    random_state = 100-8, # for reproducibility
)

In [ ]:
# Run dimensionality reduction
coordinates = reducer.fit_transform(np.asarray(vectors_to_plot))

# Store the results in a DataFrame (for convenient plotting -> next step)
df = pd.DataFrame({"word":words_to_plot})
df["x"] = coordinates[:, 0]
df["y"] = coordinates[:, 1]

We have selected words and run dimensionality reduction to get 2D coordinates for each word, we can now represent them on a planar figure:

In [ ]:
plt.scatter(
    df["x"], df["y"],
    alpha=.4
)
plt.show()

But this is not so informative as is... so let's make it interactive to be able to explore it and see what are the words associated with each dot:

In [ ]:
# Make an interactive plot
zoom = alt.selection_interval(bind="scales")

points = (
    alt.Chart(df)
    .mark_circle(size=55, opacity=0.45, color='slateblue')
    .encode(
        x=alt.X("x:Q", title=None),
        y=alt.Y("y:Q", title=None),
        tooltip=[
            alt.Tooltip("word:N", title="word"),
        ],
    )
)

chart = (
    (points)
    .add_params(zoom)
    .properties(
        width=750,
        height=600,
        title=alt.TitleParams(
            text=f"Visualisation of {len(words_to_plot)} words from “{timeslice_to_plot}” vectors.",
            fontSize=28,
            anchor="start",
        ),
    )
    .configure_axis(
        grid=True,
        domain=True,
        ticks=True,
        labels=True,
    )
    .configure_view(
        strokeWidth=0,
    )
    .configure_title(
        font="sans-serif",
    )
)

chart

**Can you find distinctive clusters on the visualisation? How is the space organised?**

You can play with the visualisation parameters and/or the list of words being shown to see how it affects the represented space.

<a name="cos_sim_1"></a>

## Cosine similarity (1)

Alright, now that we are familiar with the data, let's try to extract some quantitative observations from it.

First:

**Compute the cosine similarity between word-vectors from different time periods.**

1- To start with, make a list of 10-20 words that (supposedly(!)) did not change (too much) meaning across time (for instance: $w=\text{monday}$). We will use these words as a 'control' group to test the models' representations.

**What would you expect a word did not change meaning, how would you expect its *environment/context* to change across time? How would it realise in the embedding space? What cosine similarity value would you expect?**

2- For each of these words, compute the cosine similarity between the word's representation at time $t$ and the same word's representation for the last time slice available (e.g., $\mathrm{cosine\ similarity}\left(\mathbf{v}_\text{monday}^{t=\text{1800s}},\mathbf{v}_\text{monday}^{t=\text{1900s}}\right)$). You can store the values in a `pandas.DataFrame` (named `df`). As a suggestion, the skeleton of a function [`compute_cosine_sims`](#compute_cosine_sims) is provided: you can implement this function, or code it your way.

3- Then you can inspect the values (for example, plot the evolution of cosine similarity for each word; compute averages over time slices, or word; or take the overall average: `df.mean().mean()`), **what do you observe?**


3- A helper function, `compare_models_representations` (defined just below), allows you to visualise word representations extracted from different models, based on a list of common words. **After you have computed the cosine similarities**, you can use this function to visualise the embeddings of different models in 2-dimensions! **Does the result of the visualisation align with the cosine similarity scores you have previously computed?**

In [ ]:
# HELPER FUNCTION - IMPLEMENTED

def compare_models_representations(
    models_to_compare:tuple, # two models' names / time slices
    embeddings_dict:dict, # dict containing the names of the models : their vecotrs
    list_of_words:list,
):
    # retrive the vectors associated with the selected words
    vectors_to_plot = [
        embeddings_dict[models_to_compare[0]][w]
        for w in list_of_words
    ]+[
        embeddings_dict[models_to_compare[1]][w]
        for w in list_of_words
    ]

    # Instantiate dimensionality reducer
    reducer = PCA(
        n_components = 2,
    )

    # Run dimensionality reduction
    coordinates = reducer.fit_transform(np.asarray(vectors_to_plot))

    # One dataframe for first time slice
    df_a = pd.DataFrame({"word":anchor_words, "decade":[models_to_compare[0]]*len(anchor_words)})
    df_a["x"] = coordinates[:len(anchor_words), 0]
    df_a["y"] = coordinates[:len(anchor_words), 1]

    # Another dataframe for second time slice
    df_b = pd.DataFrame({"word":anchor_words, "decade":[models_to_compare[1]]*len(anchor_words)})
    df_b["x"] = coordinates[len(anchor_words):, 0]
    df_b["y"] = coordinates[len(anchor_words):, 1]

    # Make an interactive plot
    zoom = alt.selection_interval(bind="scales")

    points_A = ( # for first model / time slice : O
        alt.Chart(df_a)
        .mark_point(size=100, opacity=0.9)
        .encode(
            x=alt.X("x:Q", title=None),
            y=alt.Y("y:Q", title=None),
            color=alt.Color("word:N", legend=alt.Legend(title="Word")),
            shape=alt.value("circle"),
            tooltip=[
                alt.Tooltip("word:N", title="word"),
                alt.Tooltip("decade:N", title="decade"),
            ],
        )
    )

    points_B = ( # for second model / time slice : □
        alt.Chart(df_b)
        .mark_point(size=100, opacity=0.9)
        .encode(
            x=alt.X("x:Q", title=None),
            y=alt.Y("y:Q", title=None),
            color=alt.Color("word:N", legend=alt.Legend(title="Word")),
            shape=alt.value("square"),
            tooltip=[
                alt.Tooltip("word:N", title="word"),
                alt.Tooltip("decade:N", title="decade"),
            ],
        )
    )

    chart = ( # combine into the chart
        (points_A + points_B)
        .add_params(zoom)
        .properties(
            width=750,
            height=600,
            title=alt.TitleParams(
                text=f"Comparison {len(anchor_words)} words betwenn “{models_to_compare[0]}” and “{models_to_compare[1]}” vectors.",
                fontSize=28,
                anchor="start",
            ),
        )
        .configure_axis(
            grid=True,
            domain=True,
            ticks=True,
            labels=True,
        )
        .configure_view(
            strokeWidth=0,
        )
        .configure_title(
            font="sans-serif",
        )
    )

    chart
    return chart

In [ ]:
# 1- Define a list of "anchor" words
# feel free to modify it!

anchor_words = [
    "day",
    "night",
    "year",
    "time",
    "water",
    "food",
    "hand",
    "head",
    "door",
    "book",
    "monday",
    "tuesday",
    "father",
    "mother",
    "land",
    "sky",
    "sun",
    "cat",
    "dog",
]

<a name="compute_cosine_sims"></a>

### Compute cosine similarities

In [ ]:
# TODO - NOT IMPLEMENTED

def cosine_similarity(wv_x, wv_y):
  """ Compute cosine similarity between vector wv_x, and wv_y """
  raise NotImplementedError("Function `cosine_similarity` is not implemented.")

def compute_cosine_sims(
    embeddings_dict:dict[KeyedVectors],
    list_of_words:list,
):
  """
  Build a DataFrame filled with same-word cosine similarities across time slices.

  Taking as input a dictionary with
    - keys: timeslices names
    - values: vector representations
  , and a list of words (for which to compute cosine similarity scores)

  For each w in list_of_words:
      For each t slice (except the last one):
          compute cosine similarity between representations of
            (w at t) and (w at last time slice)
          store the results
  Make a DataFrame with each word as a column, each row represent one time slice
    values are the cosine similarity between that time slice (row) and the last one, for that word (column)
  """

  raise NotImplementedError("Function `compute_cosine_sims` is not implemented.")

In [ ]:
# /!\ if you did not define this function, feel free to replace it with your code
# otherwise, this cell should run (once the functions are implemented)

df = compute_cosine_sims(
    embeddings_dict=loaded_kvs,
    list_of_words=anchor_words,
)

mean_cos_sim = df[anchor_words].mean().mean()

print("-"*25+f"\nThe overall average cosine similarity across time periods and words and is: {mean_cos_sim:.3f}.")

Feel free to extand your exploration of cosine similarities!

### Visualising words from different time periods

In [ ]:
models_to_compare = ("1880s", "1900s")

In [ ]:
compare_models_representations(
    models_to_compare=models_to_compare,
    embeddings_dict=loaded_kvs,
    list_of_words=anchor_words,
)

What do you think about this plot? Does it make sense? Can you see any pattern? Within time-period? For word pairs across time periods?

Do we need to do anything else to (visually) compare words across time periods?

Note: The original data is 200-dimensional, the transformation may also blur the geometry...

Depending on your list of words, the results you got may vary, but I would guess that, your experiments showed that **average cosine similarity scores across time periods are pretty low**, typically $\leq 0.01$ (as a reminder, scores of $0$ mean perpendicular (i.e., unrelated) vectors)...

...but that is perfectly normal!

<a name="align"></a>

# Alignment Process

> 💡 **Reminder:** Word2Vec algorithms rely on *stochastic optimisation* (think random vector initialisation and iterative parameters update through negative sampling) to map words into a vector space.
>
> The algorithm optimises a *geometry* (relationships between items), *not a coordinate system* (absolute dimensions $\hat{e_1},\ \hat{e_2},\ \hat{e_3},\cdots,\hat{e_d}$).
> In other words, because of stochasticity, running Word2Vec twice (even on the exact same corpus) produces vector spaces that may represent congruent geometries, but that are rotated relative to one another.
>
> Hence, the representation spaces of independently trained `Word2Vec` models first need to be *aligned* before they can be compared across time-slices.


---

<details><summary><b>Orthogonal Procrustes Alignment</b> (detailed closed form solution)</summary>

The goal of the orthogonal Procrustes problem is to find the orthogonal transformation matrix $W$ which most closely maps a given matrix $X$ to a reference matrix $Y$, with $X,Y\in\mathbb{R}^{n\times d}$.

Formally, we want to find $W^\star = \underset{W}{\mathrm{min}}\lVert XW-Y \rVert_{F}$, subject to $WW^T=I$.
The optimization of the Frobenius norm of the difference between $XW$ (transformed target) and $Y$ (reference) makes the aligned vector of each word (rows of the matrices) collectively as close as possible to its counterpart in the reference space.

This minimization problem is equivalent to maximizing the trace of the matrix $W^TX^TY$, indeed:

\begin{align}
\lVert XW-Y \rVert_{F}^2 :&= \mathrm{tr}\left( (XW-Y)^T(XW-Y) \right) \\
&= \mathrm{tr}\left((W^TX^T-Y^T) (XW-Y)\right) \\
&= \mathrm{tr}\left(W^TX^TXW\right)+\mathrm{tr}\left(Y^TY\right)- \mathrm{tr}\left( W^TX^TY\right) - \mathrm{tr}\left(Y^TXW \right) \\
&\stackrel{(a)}{=} \mathrm{tr}\left(X^TX\right)+\mathrm{tr}\left(Y^TY\right)- \mathrm{tr}\left( W^TX^TY\right) - \mathrm{tr}\left(Y^TXW \right) \\
&\stackrel{(b)}{=} \mathrm{tr}\left(X^TX\right)+\mathrm{tr}\left(Y^TY\right)- 2 \mathrm{tr}\left( W^TX^TY\right) \\
\end{align}

where:
- $(a)$ uses cyclic permutation $\mathrm{tr}(AB)=\mathrm{tr}(BA)$, and the orthogonality constraint $W^TW=I$;
- $(b)$ $\mathrm{tr}(A)=\mathrm{tr}(A^T)$

Since the first two terms ($\mathrm{tr}\left(X^TX\right), \mathrm{tr}\left(Y^TY\right)$ are fixed, the minimisation problem is equivalent to maximizing $\mathrm{tr}\left( W^TX^TY\right)$.

Then, to solve this problem, let's introduce $M=X^TY\in\mathbb{R}^{d\times d}$.
This matrix $M$ can be decomposed through [singular value decomposition](https://en.wikipedia.org/wiki/Singular_value_decomposition) into: $M=U\Sigma V^T$, where $U,V\in\mathbb{R}^{d\times d}$ are orthogonal matrices, and $\Sigma=\mathrm{diag}(\sigma_1,\cdots,\sigma_d)$ is a diagonal matrix filled with the singular values of $M$.

Now, using this decomposition, the objective is to find $W$ that maximizes:

\begin{align}
\mathrm{tr}(W^TM) &= \mathrm{tr}(W^TU\Sigma V^T) \\
&\stackrel{(c)}{=} \mathrm{tr}(R\Sigma) \\
&\stackrel{(d)}{=} \sum_{i=1}^d \sigma_i R_{ii}
\end{align}

where:
- $(c)$ use $\mathrm{tr}(AB)=\mathrm{tr}(BA)$; and defines the orthogonal matrix $R:=V^TQ^TU$, indeed $RR^T=(V^TW^TU)(U^TWV)=(V^TW^T(UU^T)WV)=I$
- $(d)$ since $\Sigma$ is diagonal

Since $R$ is orthogonal by construction, we have $R_{ii}\in[-1;1]$. With this constraint, the earlier expression is maximised for $R_{ii}=1,\ \forall i\in\{1,\cdots,d\}$ (because $\sigma_i\geq 0$).

Thus, to maximise this expression, we must have $R=I$, so:

\begin{align}
R &= I \\
\Leftrightarrow V^TW^TU &= I \\
\Leftrightarrow V(V^TW^TU)U^T &= VIU^T \\
\Leftrightarrow (VV^T)W^T(UU^T) &= VU^T \\
\Leftrightarrow W^T &= VU^T \\
\Leftrightarrow W &= UV^T \\
\end{align}

This proof gives us a closed form solution for the orthogonal transformation that minimises the overall distance between the target and the reference systems:

$$\boxed{W^\star = UV^T}$$

where $U,V\in\mathbb{R}^{d\times d}$ are obtained from the SVD of $M=X^TY=U\Sigma V^T$.

</details>

**Complete the function below to perform orthogonal Procrustes alignment using the last time-slice as reference (or base), and directly return the aligned vectors.**

Two pre-implemented functions make the broad labour (ensuring that we align the representations based on a shared vocabulary, and taking care of the normalisation process). **You need to implement the `find_procrustes_solution` to find the orthogonal matrix to map 'other' vectors onto 'base' vectors' space.**

Remember, the *Orthogonal Procrustes Problem*: $\underset{W}{\mathrm{min}}\lVert\mathbf{X}\mathbf{W}-\mathbf{Y}\rVert_F$ has a closed form solution given by:


$$\boxed{W^\star = UV^T}$$

where $U,V\in\mathbb{R}^{d\times d}$ are obtained from the SVD of $M=X^TY=U\Sigma V^T$.

> 💡 In python you can run SVD decomposition using `np.linalg.svd`, which returns the tuple ($\mathbf{U}, \mathbf{\Sigma}, \mathbf{V}^T$), i.e., `U, _, Vt = np.linalg.svd(M)`.

In [ ]:
# TODO - NOT IMPLEMENTED

def find_procrustes_solution(base_vecs, other_vecs):
    """ Find the Procrustes transform to align vectors from others to base """

    # TODO IMPLEMENT THE FUNCTION:
    # Steps:
    # 1. Obtain matrix M
    # 2. Solve SVD
    # 3. Construct martix W*
    # -> return the matrix W*

    raise NotImplementedError("You need to implement the function `find_procrustes_solution` first!")

In [ ]:
# Already implemented, code (slightly modified) from: https://github.com/Living-with-machines/DiachronicEmb-BigHistData/blob/main/scripts/training/utils.py$0

def smart_procrustes_align_gensim_from_kv(base_embed_kv, other_embed_kv, words=None):
    """
    Original script: https://gist.github.com/quadrismegistus/09a93e219a6ffc4f216fb85235535faf
    Procrustes align two gensim word2vec models (to allow for comparison between same word across models).
    Code ported from HistWords <https://github.com/williamleif/histwords> by William Hamilton <wleif@stanford.edu>.

    First, intersect the vocabularies (see `intersection_align_gensim` documentation).
    Then do the alignment on the other_embed model.
    Replace the other_embed model's syn0 and syn0norm numpy matrices with the aligned version.
    Return other_embed.

    If `words` is set, intersect the two models' vocabulary with the vocabulary in words (see `intersection_align_gensim` documentation).
    """

    # patch by Richard So [https://twitter.com/richardjeanso) (thanks!) to update this code for new version of gensim

    # make sure vocabulary and indices are aligned
    in_base_embed_kv, in_other_embed_kv = intersection_align_gensim_from_kv(base_embed_kv, other_embed_kv, words=words)

    # re-filling the normed vectors
    in_base_embed_kv.fill_norms(force=True)
    in_other_embed_kv.fill_norms(force=True)

    # get the (normalized) embedding matrices
    base_vecs = in_base_embed_kv.get_normed_vectors()
    other_vecs = in_other_embed_kv.get_normed_vectors()

    # Solve Procrustes
    ortho = find_procrustes_solution(base_vecs, other_vecs)

    # Replace original array with modified one, i.e. multiplying the embedding matrix by "ortho"
    other_embed_kv.vectors = (other_embed_kv.vectors).dot(ortho)

    return other_embed_kv

# ------------------- intersection_align_gensim --------------------
# from: https://gist.github.com/quadrismegistus/09a93e219a6ffc4f216fb85235535faf
def intersection_align_gensim_from_kv(m1_kv, m2_kv, words=None):
    """
    Intersect two gensim word2vec models, m1 and m2.
    Only the shared vocabulary between them is kept.
    If 'words' is set (as list or set), then the vocabulary is intersected with this list as well.
    Indices are re-organized from 0..N in order of descending frequency (=sum of counts from both m1 and m2).
    These indices correspond to the new syn0 and syn0norm objects in both gensim models:
        -- so that Row 0 of m1.syn0 will be for the same word as Row 0 of m2.syn0
        -- you can find the index of any word on the .index2word list: model.index2word.index(word) => 2
    The .vocab dictionary is also updated for each model, preserving the count but updating the index.
    """

    # Get the vocab for each model
    vocab_m1 = set(m1_kv.index_to_key)
    vocab_m2 = set(m2_kv.index_to_key)

    # Find the common vocabulary
    common_vocab = vocab_m1 & vocab_m2
    if words: common_vocab &= set(words)

    # If no alignment necessary because vocab is identical...
    if not vocab_m1 - common_vocab and not vocab_m2 - common_vocab:
        return (m1_kv,m2_kv)

    # Otherwise sort by frequency (summed for both)
    common_vocab = list(common_vocab)

    # Then for each model...
    for m_kv in [m1_kv, m2_kv]:
        # Replace old syn0norm array with new one (with common vocab)
        indices = [m_kv.key_to_index[w] for w in common_vocab]
        old_arr = m_kv.vectors
        new_arr = np.array([old_arr[index] for index in indices])
        m_kv.vectors = new_arr

        # Replace old vocab dictionary with new one (with common vocab)
        # and old index2word with new one
        new_key_to_index = {}
        new_index_to_key = []
        for new_index, key in enumerate(common_vocab):
            new_key_to_index[key] = new_index
            new_index_to_key.append(key)
        m_kv.key_to_index = new_key_to_index
        m_kv.index_to_key = new_index_to_key

    return (m1_kv,m2_kv)

In [ ]:
# Run the procrustes alignment and store the aligned vectors!

kvs_aligned = {}

for y in decades[:-1]: # we 'align' with representations from the last timeslice
    kvs_aligned[y] = smart_procrustes_align_gensim_from_kv(
        base_embed_kv=loaded_kvs[decades[-1]],
        other_embed_kv=loaded_kvs[y],
        words=None
    )

kvs_aligned[decades[-1]] = loaded_kvs[decades[-1]]

### Cosine similarity (2)

Alright, so now that we have run the orthogonal Procrustes transform to align all of our representations with the last one. Meaning, all representations pair should be aligned!

So, let's replicate your cosine similarity analysis, but using the **aligned representations** this time!

**Do you expect any difference?**

In [ ]:
# if you did not define this function, feel free to replace it with your code
# otherwise, this should run

df = compute_cosine_sims(
    kv_dict=kvs_aligned, # we switched this value with our aligned vector!
    list_of_words=anchor_words,
)

mean_cos_sim = df[anchor_words].mean().mean()

print("-"*25+f"\nThe overall average cosine similarity across time periods and words and is: {mean_cos_sim:.3f}.")

Compare this new value with the one obtained previously, what do you think?

We can also visualise the representations for a set of pre-defined words, as previously:

In [ ]:
compare_models_representations(
    models_to_compare=models_to_compare,
    embeddings_dict=kvs_aligned, # switched to the aligned representations!
    list_of_words=anchor_words,
)

**How did the representations change? Do you observe different patterns than earlier? Does it seem coherent with the *Procrustes alignment* we computed?**

<a name="analysis"></a>

# Analysis

Now that our vector spaces across all decades are aligned (and that we have verified it worked!), it is your time to shine ✨

Up until now, you have seen how the technical pipeline works. In this section, your goal is to formulate a small research question or exploration strategy, run your computational analysis, and interpret the findings in light of 19th-century history.

You can jump directly [right in](#your_space_here), start by using the [visualiser helper](#make_drift_visualisation) described just below, or review [some ideas](#analysis_ideas) to get inspired.

---



## 👀 Helper Tool: Interactive Semantic Drift Visualization

To help you get started with qualitative exploration, we have provided the [`make_drift_visualisation`](#make_drift_visualisation) function.

For any target `keyword`, this function:
1. Identifies its nearest neighbors in each decade slice.
2. Extracts their vector representations from the reference space (the most recent model).
3. Tracks the trajectory of your keyword across time using 2D dimensionality reduction (by default, t-SNE).

The output is an interactive scatter plot where you can observe how a word moves through semantic space over the century.
Note: this is function used to make the [`'train'` plot](#train_plot) at the top of this notebook 🚂:

```python
make_drift_visualisation(
    "train", # the keyword of your choice here!
    embeddings_dict=kvs_aligned,
)
```

Feel free to use this function to explore the data and to get interactive visualisations of *semantic drifts* !

In [ ]:
def _get_words_to_plot(
    keyword,
    embeddings_dict,
    timeslices,
    topn=20,
    additional_words=None,
    words_to_remove=None,
    verbose=True,
):
    """Collect the words to display around the keyword."""
    words = set()

    for t in timeslices:
        if keyword not in embeddings_dict[t]:
            raise KeyError(f"Keyword: '{keyword}' not present in representations from '{t}'.")
        neighbours = embeddings_dict[t].most_similar(keyword, topn=topn) # format: list[tuple], with tuple: (word, similarity score)

        if verbose:
            print(f"{t}: found {len(neighbours)} neighbours:\n\t{', '.join([w for w, _ in neighbours])}")

        words.update(word for word, _ in neighbours)

    # Add user-specified words
    if additional_words:
        words.update(additional_words)

    # Remove unwanted words
    if words_to_remove:
        words.difference_update(words_to_remove)

    # The keyword itself is represented separately
    words.discard(keyword)

    return sorted(words)

def _get_similarities(
    keyword,
    context_word,
    embeddings_dict,
    timeslices,
):
    """Compute similarity across time slices"""
    return {
        f"similarity_{t}": embeddings_dict[t].similarity(keyword, context_word)
        for t in timeslices
    }

def _prepare_drift_data(
    keyword,
    embeddings_dict,
    timeslices,
    words,
    reducer,
):
    """
    Create a DataFrame containing 2D coordinates and metadata.

    Neighbouring words are represented using the reference (last) time slice.
    The keyword is represented once per time slice.
    """

    reference_time = timeslices[-1]
    vectors = []
    metadata = []

    # Neighbouring words
    for word in words:
        if word not in embeddings_dict[reference_time]:
            continue

        vectors.append(embeddings_dict[reference_time][word])
        metadata.append({
            "word": word,
            "type": "Neighbour",
            "timeslice": str(reference_time),
        } | _get_similarities( # get similarity across time periods
                keyword=keyword,
                context_word=word,
                embeddings_dict=embeddings_dict,
                timeslices=timeslices
            )
        )

    # Keyword across time
    for t in timeslices:
        if keyword not in embeddings_dict[t]:
            continue

        vectors.append(embeddings_dict[t][keyword])
        metadata.append({
            "word": keyword,
            "type": "Keyword",
            "timeslice": str(t),
        } | _get_similarities( # get similarity across time periods
                keyword=keyword,
                context_word=word,
                embeddings_dict=embeddings_dict,
                timeslices=timeslices
            )
        )

    if len(vectors) < 2:
        raise ValueError(
            "Not enough words/vectors were found to perform dimensionality reduction."
        )

    # Reduce embeddings to two dimensions
    coordinates = reducer.fit_transform(np.asarray(vectors))

    df = pd.DataFrame(metadata)
    df["x"] = coordinates[:, 0]
    df["y"] = coordinates[:, 1]

    return df

<a name="make_drift_visualisation"></a>

In [ ]:
def make_drift_visualisation(
    keyword,
    embeddings_dict,
    timeslices_to_plot=None,
    topn=20,
    additional_words=None,
    words_to_remove=None,
    dimensionality_reducer=None,
    verbose=True,
    save_fig_name=None,
):
    """
    Visualise the semantic trajectory of a keyword across time.

    Neighbouring words are taken from the final/reference time slice.
    The keyword is plotted once for each time slice.

    Parameters
    ----------
    keyword : str
        Word whose semantic drift is being visualised.

    embeddings_dict : dict
        Dictionary mapping time slices to gensim KeyedVectors.

    timeslices_to_plot : list, optional
        Time slices to include. Defaults to all available slices.

    topn : int
        Number of nearest neighbours to retrieve per time slice.

    additional_words : list, optional
        Extra words to include in the visualisation.

    words_to_remove : list, optional
        Words to exclude from the visualisation.

    dimensionality_reducer : object, optional
        Object with a ``fit_transform`` method. Defaults to t-SNE.

    verbose : bool
        Print progress information.

    save_fig_name : str, optional
        If provided, save the Altair chart as an HTML file.
    """

    # 1. Validate and prepare inputs

    if not embeddings_dict:
        raise ValueError("embeddings_dict is empty.")

    if timeslices_to_plot is None:
        timeslices_to_plot = sorted(embeddings_dict.keys())
    else:
        timeslices_to_plot = list(timeslices_to_plot)

    missing_timeslices = [
        t for t in timeslices_to_plot if t not in embeddings_dict
    ]

    if missing_timeslices:
        raise ValueError(
            f"Unknown timeslices: {missing_timeslices}"
        )

    if not timeslices_to_plot:
        raise ValueError("No timeslices were selected.")

    if topn < 1:
        raise ValueError("topn must be at least 1.")

    # Create a reducer if none given, default: TSNE with following params
    if dimensionality_reducer is None:
        dimensionality_reducer = TSNE(
            n_components=2,
            random_state=92,
            metric="euclidean",
            learning_rate="auto",
            perplexity=5,
        )

    # 2. Find words to display
    if verbose:
        print("Finding neighbouring words...")

    words = _get_words_to_plot(
        keyword=keyword,
        embeddings_dict=embeddings_dict,
        timeslices=timeslices_to_plot,
        topn=topn,
        additional_words=additional_words,
        words_to_remove=words_to_remove,
        verbose=verbose,
    )

    if verbose:
        print(f"Words selected: {len(words)}")

    # 3. Prepare coordinates
    if verbose:
        print("Computing metadata and 2D representation...")

    df = _prepare_drift_data(
        keyword=keyword,
        embeddings_dict=embeddings_dict,
        timeslices=timeslices_to_plot,
        words=words,
        reducer=dimensionality_reducer,
    )

    # 4. Build the Altair visualisation
    neighbours = df[df["type"] == "Neighbour"]
    trajectory = df[df["type"] == "Keyword"]

    # Interactive zoom / pan
    zoom = alt.selection_interval(bind="scales")

    # some params
    fontsize = 16

    # Neighbouring words
    points = (
        alt.Chart(neighbours)
        .mark_circle(size=55, opacity=0.45, color='grey')
        .encode(
            x=alt.X("x:Q", title=None),
            y=alt.Y("y:Q", title=None),
            tooltip=[
                alt.Tooltip("word:N", title="Word"),
            ]+[
                alt.Tooltip(f"similarity_{t}", title=f"Similarity in {t}", format=".2f")
                for t in timeslices_to_plot
            ],
        )
    )

    labels = (
        alt.Chart(neighbours)
        .mark_text(
            align="left",
            dx=7,
            dy=-3,
            fontSize=fontsize,
            opacity=0.65,
        )
        .encode(
            x="x:Q",
            y="y:Q",
            text="word:N",
        )
    )

    # Keyword trajectory
    line = (
        alt.Chart(trajectory)
        .mark_line(
            strokeWidth=2.5,
            point=True,
            color="crimson"
        )
        .encode(
            x="x:Q",
            y="y:Q",
            order="timeslice:N",
            tooltip=[
                alt.Tooltip("word:N", title="Keyword"),
                alt.Tooltip("timeslice:N", title="Period"),
            ],
        )
    )

    keyword_points = (
        alt.Chart(trajectory)
        .mark_circle(size=180, color="crimson")
        .encode(
            x="x:Q",
            y="y:Q",
            tooltip=[
                alt.Tooltip("word:N", title="Keyword"),
                alt.Tooltip("timeslice:N", title="Period"),
            ],
        )
    )

    keyword_labels = (
        alt.Chart(trajectory)
        .mark_text(
            dy=-14,
            fontSize=fontsize*1.2,
            fontWeight="bold",
        )
        .encode(
            x="x:Q",
            y="y:Q",
            text="timeslice:N",
        )
    )

    chart = (
        (points + labels + line + keyword_points + keyword_labels)
        .add_params(zoom)
        .properties(
            width=750,
            height=600,
            title=alt.TitleParams(
                text=f"Semantic drift visualisation for “{keyword}”",
                fontSize=fontsize*2,
                anchor="start",
            ),
        )
        .configure_axis(
            grid=False,
            domain=False,
            ticks=False,
            labels=False,
        )
        .configure_view(
            strokeWidth=0,
        )
        .configure_title(
            font="sans-serif",
        )
    )

    # 5. Save / display
    if save_fig_name:
        if verbose:
            print(f"Saving figure to {save_fig_name}...")

        chart.save(save_fig_name)

    return chart

---

<a name="analysis_ideas"></a>

## Suggested Analysis Ideas

You can structure your analysis along one of the following tracks, blend them according to your taste, or just go on your own custom path.

**Write a Word’s "Biography"**
* **Goal:** Track the 'semantic life' of a (or a collection of) key term(s) of your choice through semantic neighbors, semantic space, time, and text.

<details><summary><b>Suggested leads</b></summary>

- Plot its trajectory across decades using `make_drift_visualisation`.
- Inspect its top nearest neighbors across time slices to pin down when and how the meaning shifts.
- Observe 'semantic shift' purely based on nearest neighbors, as an alternative to semantic space alignment, you can even compute a score based on the nearest neighbors for instance as in [(Gonen et al., 2020)](https://aclanthology.org/2020.acl-main.51)
- Validate your computational findings against your own historical knowledge and/or external sources: check dictionary entries (e.g., OED historical first records) or go back to the original newspaper context to see if the shift reflects genuine semantic change or thematic news bias.

</details>


**Automatic Shift Detection**
* **Goal:** Let the algorithms find the biggest shifters for you instead of guessing words in advance.


<details><summary><b>Suggested leads</b></summary>

- Compute cosine similarity between every word's 1800s vector and its aligned 1910s vector.
- Rank terms by lowest similarity (greatest movement).
- Alternatively, inspect highest rate of change in NN
- Compare the results
- **What are the top words?**

</details>

**Test "Statistical Laws" of Semantic Change**
* **Goal:** Evaluate broader hypotheses about how language evolves over time. See: *Diachronic Word Embeddings Reveal Statistical Laws of Semantic Change* [(Hamilton, Leskovec, and Jurafsky, 2016)](https://aclanthology.org/2020.acl-main.51/).

<details><summary><b>Suggested leads</b></summary>

- Pick a hypothesis (e.g., *Law of Conformity*: high-frequency words shift slower; or *Law of Polysemy*: broad words shift faster).
- Use the aligned vector spaces and word frequencies to test whether the data supports the theory.

</details>

---

---

---

---

---

<a name="your_space_here"></a>